# Query Pipeline — COSORA

Búsqueda híbrida (Dense + BM25 + RRF), prompt builder y generación LLM.

**Prerequisito:** ejecutar `ingestion_pipeline.ipynb` con el mismo `RUNTIME` y `EMBED_BACKEND`.

Para benchmark MrBERT vs E5: ingesta con `INDEX_BOTH=True`, luego sección 6.

> **Nota:** Este notebook es auto-contenido. Funciona tanto en local como en Colab sin necesidad de subir archivos `.py` adicionales.

## 0. Setup

Instalación de dependencias, auto-detección local/Colab, y código COSORA embebido.

In [12]:
%pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv nltk pandas torch

import nltk
nltk.download('punkt', quiet=True)

Note: you may need to restart the kernel to use updated packages.


True

In [27]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-DETECT: Colab vs Local  +  Configuración
# ═══════════════════════════════════════════════════════════════════════
import glob
import os
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path
from typing import Any

import chromadb
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

IN_COLAB = "google.colab" in sys.modules

# ─── Configuración ────────────────────────────────────────────────────
RUNTIME = "auto"            # "auto" | "colab" | "local"
EMBED_BACKEND = "e5"    # "mrbert" | "e5" — debe coincidir con la ingesta

GEN_BACKEND = "openai"       # "local" | "openai"
LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

RETRIEVAL_K = 100
TOP_N = 10
RRF_K = 60
RRF_MIN_SCORE = 0.015
SCORE_RATIO = 0.60

if RUNTIME == "auto":
    RUNTIME = "colab" if IN_COLAB else "local"

if RUNTIME == "colab" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

def _find_project_root() -> Path:
    """Walk up from CWD until a dir containing .env or data/raw is found."""
    candidate = Path(".").resolve()
    for _ in range(6):
        if (candidate / ".env").exists() or (candidate / "data" / "raw").exists():
            return candidate
        parent = candidate.parent
        if parent == candidate:
            break
        candidate = parent
    return Path(".").resolve()

if GEN_BACKEND == "openai":
    from dotenv import load_dotenv
    if RUNTIME == "colab":
        load_dotenv("/content/drive/MyDrive/variablentorno/.env")
    else:
        load_dotenv(_find_project_root() / ".env")

# ─── Rutas ────────────────────────────────────────────────────────────
def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
    else:
        root = _find_project_root()
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(root / "notebooks" / "chroma_db")
    return docs_dir, chroma_path

# ═══════════════════════════════════════════════════════════════════════
# COSORA SHARED — código embebido (no requiere cosora_shared.py)
# ═══════════════════════════════════════════════════════════════════════

EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "mrbert": {
        "model_id": "BSC-LT/MrBERT-es",
        "collection": "cosora_chunks_mrbert",
        "type": "transformers_cls",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}

TECH_TOKEN_PATTERN = re.compile(
    r"^(AR-\d+|UTE|DF|DEO|DO|PPI|[A-Z]{2,}\d*|\d+[A-Z-]+)$",
    re.IGNORECASE,
)

STOPWORDS_ES = {
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por", "un",
    "para", "con", "no", "una", "su", "al", "es", "lo", "como", "más", "pero", "sus",
    "le", "ya", "o", "este", "sí", "porque", "esta", "entre", "cuando", "muy", "sin",
    "sobre", "también", "me", "hasta", "hay", "donde", "quien", "desde", "todo", "nos",
    "durante", "todos", "uno", "les", "ni", "contra", "otros", "ese", "eso", "ante",
    "ellos", "e", "esto", "mí", "antes", "algunos", "qué", "unos", "yo", "otro", "otras",
    "otra", "él", "tanto", "esa", "estos", "mucho", "quienes", "nada", "muchos", "cual",
    "poco", "ella", "estar", "estas", "algunas", "algo", "nosotros", "mi", "mis", "tú",
    "te", "ti", "tu", "tus", "ellas", "nosotras", "vosotros", "vosotras", "os",
}


def backend_config(backend):
    if backend not in EMBED_BACKENDS:
        raise ValueError(f"EMBED_BACKEND desconocido: {backend}. Usa: {list(EMBED_BACKENDS)}")
    return EMBED_BACKENDS[backend]


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self.cfg["type"] == "sentence_transformers":
            if self._st_model is None:
                from sentence_transformers import SentenceTransformer
                self._st_model = SentenceTransformer(self.cfg["model_id"])
            return
        if self._model is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.cfg["model_id"])
            self._model = AutoModel.from_pretrained(self.cfg["model_id"])
            self._model.eval()

    def embed_one(self, text, *, is_query=False):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            vec = self._st_model.encode(prefix + text)
            return vec.tolist() if hasattr(vec, "tolist") else list(vec)
        assert self._tokenizer is not None and self._model is not None
        inputs = self._tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self._model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    def embed_batch(self, texts, *, is_query=False, batch_size=16):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            prefixed = [prefix + t for t in texts]
            vecs = self._st_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True)
            return vecs.tolist()
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            for text in batch:
                all_embeddings.append(self.embed_one(text, is_query=is_query))
        return all_embeddings


def strip_doc_prefix(text, backend):
    prefix = backend_config(backend).get("doc_prefix", "")
    if prefix and text.startswith(prefix):
        return text[len(prefix):]
    return text


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def tokenize_bm25(text, stemmer=None):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = [w for w in text.split() if w and w not in STOPWORDS_ES]
    if stemmer is None:
        return words
    result = []
    for w in words:
        if TECH_TOKEN_PATTERN.match(w):
            result.append(w)
        else:
            result.append(stemmer.stem(w))
    return result


def build_bm25_index(collection):
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer("spanish")
    except Exception:
        stemmer = None
    all_data = collection.get()
    all_docs = all_data["documents"]
    all_metas = all_data["metadatas"]
    tokenized = [tokenize_bm25(doc, stemmer) for doc in all_docs]
    return BM25Okapi(tokenized), all_docs, all_metas, stemmer


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]


def bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=100):
    scores = bm25_index.get_scores(tokenize_bm25(query, stemmer))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"text": all_docs[i], "meta": all_metas[i], "rank_bm25": rank}
        for rank, i in enumerate(top_idx)
    ]


def rrf_fusion(dense, bm25, rrf_k=60, top_n=10):
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return ranked[:top_n]


def retrieve_hybrid(query, backend, chroma_path, retrieval_k=100, top_n=10, rrf_k=60, embedder=None):
    collection, _ = get_chroma_collection(chroma_path, backend)
    if embedder is None or embedder.backend != backend:
        embedder = Embedder(backend)
    bm25_idx, docs, metas, stemmer = build_bm25_index(collection)
    d = dense_search(collection, embedder, query, k=retrieval_k)
    b = bm25_search(query, bm25_idx, docs, metas, stemmer, k=retrieval_k)
    return rrf_fusion(d, b, rrf_k=rrf_k, top_n=top_n)

# ─── Inicializar ─────────────────────────────────────────────────────
DOCS_DIR, CHROMA_PATH = resolve_paths(RUNTIME)
collection, cfg = get_chroma_collection(CHROMA_PATH, EMBED_BACKEND)
embedder = Embedder(EMBED_BACKEND)
bm25_index, all_docs, all_metas, stemmer = build_bm25_index(collection)

print(f"IN_COLAB={IN_COLAB}  RUNTIME={RUNTIME}")
print(f"EMBED_BACKEND={EMBED_BACKEND}")
print(f"Chroma: {cfg['collection']} ({collection.count()} chunks)")
print(f"GEN_BACKEND={GEN_BACKEND}")
print("\n✅ Código COSORA cargado correctamente")

IN_COLAB=False  RUNTIME=local
EMBED_BACKEND=e5
Chroma: cosora_chunks_e5 (799 chunks)
GEN_BACKEND=openai

✅ Código COSORA cargado correctamente


## 1. Retrieval híbrido

In [14]:
def retrieve_for_backend(query, backend, top_n=TOP_N):
    return retrieve_hybrid(
        query,
        backend,
        CHROMA_PATH,
        retrieval_k=RETRIEVAL_K,
        top_n=top_n,
        rrf_k=RRF_K,
    )


def retrieve_active(query, top_n=TOP_N):
    dense = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    bm25 = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    return rrf_fusion(dense, bm25, rrf_k=RRF_K, top_n=top_n)

## 2. Prompt Builder

In [15]:
def build_prompt(query, chunks):
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        doc_id = chunk["meta"]["doc_id"]
        text = strip_doc_prefix(chunk["text"], EMBED_BACKEND)
        context_blocks.append(f"[Fragmento {i} - Fuente: {doc_id}]\n{text}")

    context = "\n\n".join(context_blocks)

    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE con información del CONTEXTO proporcionado.
2. Si la información no está en el contexto, dilo explícitamente.
3. Cita siempre la fuente entre paréntesis: (Fuente: nombre_del_acta).
4. Si varios fragmentos hablan del mismo tema, sintetiza la información en una respuesta coherente, no repitas datos.

FORMATO DE RESPUESTA:
- Usa puntos numerados cuando haya varios aspectos.
- Prioriza: fechas, responsables (DF, UTE, constructora), decisiones tomadas y acciones pendientes.
- Si detectas una evolución temporal (algo cambió entre actas), indícalo cronológicamente.

VOCABULARIO:
- DF = Dirección Facultativa
- UTE = Unión Temporal de Empresas (constructora)
- DEO = Director de Ejecución de Obra
- DO = Director de Obra

=== CONTEXTO ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""

## 3. Generador LLM

In [16]:
import torch
from transformers import pipeline

generator = None
if GEN_BACKEND == "local":
    print(f"Cargando {LOCAL_MODEL}...")
    generator = pipeline(
        "text-generation",
        model=LOCAL_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    print("Modelo cargado")


def generate_answer(prompt):
    if GEN_BACKEND == "local":
        output = generator(
            prompt,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        return output[0]["generated_text"][len(prompt) :].strip()

    if GEN_BACKEND == "openai":
        from openai import OpenAI

        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY no encontrada en .env")
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content

    raise ValueError(f"GEN_BACKEND desconocido: {GEN_BACKEND}")

## 4. Pipeline `ask_cosora`

In [17]:
def filter_chunks(chunks):
    if not chunks or chunks[0]["score"] < RRF_MIN_SCORE:
        return None
    max_score = chunks[0]["score"]
    return [c for c in chunks if c["score"] >= max_score * SCORE_RATIO]


def ask_cosora(query, verbose=True, backend=None):
    backend = backend or EMBED_BACKEND
    if backend == EMBED_BACKEND:
        chunks = retrieve_active(query, top_n=TOP_N)
    else:
        chunks = retrieve_for_backend(query, backend, top_n=TOP_N)

    filtered = filter_chunks(chunks)
    if filtered is None:
        msg = "No se encontró información suficientemente relevante en las actas."
        if verbose:
            print(f"Query: {query}\n{msg}")
        return msg

    prompt = build_prompt(query, filtered)
    answer = generate_answer(prompt)

    if verbose:
        print(f"Query: {query} [backend={backend}]")
        print(f"Chunks ({len(filtered)}):")
        for c in filtered:
            t = strip_doc_prefix(c["text"], backend)[:80]
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f} | {t}...")
        print(f"\nRespuesta ({GEN_BACKEND}):\n{answer}\n" + "-" * 70)

    return answer

## 4b. Query Rewriting

Para cada pregunta del usuario se generan **3 versiones mejoradas** via GPT que potencian keywords técnicos y variaciones semánticas. Luego se hace retrieval híbrido con las 4 queries (original + 3 reescritas) y se fusionan los resultados con RRF de segundo nivel.

| Función | Qué hace |
|---------|----------|
| `rewrite_query(q)` | Llama a GPT-4o-mini y devuelve 3 queries mejoradas |
| `retrieve_multi_query(queries)` | Retrieval híbrido por cada query + RRF de 2º nivel |
| `ask_cosora_rewritten(q)` | Pipeline completo: rewriting → multi-retrieval → LLM → comparación |

In [18]:
# ── Query Rewriting ───────────────────────────────────────────────────

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def rewrite_query(query: str, n: int = 3) -> list[str]:
    """Genera n versiones mejoradas de la query via GPT-4o-mini."""
    prompt = f"""Eres un experto en sistemas RAG aplicados a actas de visita semanal de obras de construcción ferroviaria española.

Tu tarea es reescribir la siguiente pregunta generando {n} variantes que mejoren la recuperación en un sistema de búsqueda híbrida (BM25 + embeddings semánticos). Cada variante cumple un rol distinto.

CONTEXTO DEL CORPUS: Actas semanales de obra para mejora de accesibilidad en estación ferroviaria (rampas, ascensores, edículos, andenes, edificio de viajeros). Roles habituales: DC, DF, DO, DEO, UTE, CSS. Elementos frecuentes: micropilotes, encofrado, hormigonado, impermeabilización, solera, arquetas, canalizaciones ADIF/RENFE, pilares metálicos HEB, pletina, losa, batache, pavimento podoTáctil.

TIPOS DE VARIANTES (genera una por tipo, en este orden):
1. KEYWORDS — Lista de términos técnicos exactos separados por comas: sustantivos, códigos de plano, materiales, roles y estados del dominio. Sin forma de pregunta.
2. EXPANSIÓN SEMÁNTICA — Reformula la pregunta con sinónimos y conceptos relacionados que capturen el mismo significado desde ángulos distintos. Útil para embeddings.
3. ESPECIFICACIÓN TÉCNICA — Versión más concreta que introduce terminología de construcción, estructura o instalaciones precisa al elemento preguntado.
4. PERSPECTIVA DE AGENTE O ESTADO — Reformula desde el punto de vista del responsable (DF, UTE, DO...) o del estado del elemento (ejecutado, pendiente, aprobado, incidencia abierta...).

PREGUNTA ORIGINAL: {query}

Devuelve exactamente {n} variantes en el orden de los tipos anteriores, una por línea, sin numeración ni prefijos. Solo en español."""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=400,
    )
    lines = [l.strip() for l in resp.choices[0].message.content.strip().split("\n") if l.strip()]
    return lines[:n]


def retrieve_multi_query(queries: list[str], top_n: int = TOP_N) -> list[dict]:
    """
    Retrieval híbrido sobre múltiples queries con RRF de segundo nivel.
    Cada query produce su ranking independiente; se fusionan todos con RRF.
    """
    scores: dict = {}
    for q in queries:
        hits = retrieve_active(q, top_n=RETRIEVAL_K)
        for rank, hit in enumerate(hits):
            cid = hit["meta"]["chunk_id"]
            if cid not in scores:
                scores[cid] = {"text": hit["text"], "meta": hit["meta"], "score": 0.0}
            scores[cid]["score"] += 1.0 / (RRF_K + rank)
    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return ranked[:top_n]


def ask_cosora_rewritten(query: str, n_rewrites: int = 3, verbose: bool = True) -> dict:
    """
    Pipeline completo con query rewriting.
    Compara retrieval con query original vs multi-query reescrito.
    """
    # 1. Generar queries reescritas
    rewritten = rewrite_query(query, n=n_rewrites)

    # 2. Retrieval: original solo
    orig_chunks = retrieve_active(query, top_n=TOP_N)
    orig_filtered = filter_chunks(orig_chunks)

    # 3. Retrieval: multi-query (original + reescritas)
    multi_chunks = retrieve_multi_query([query] + rewritten, top_n=TOP_N)
    multi_filtered = filter_chunks(multi_chunks)

    # 4. Generar respuestas
    orig_answer = (
        generate_answer(build_prompt(query, orig_filtered))
        if orig_filtered else "No se encontró información suficientemente relevante."
    )
    multi_answer = (
        generate_answer(build_prompt(query, multi_filtered))
        if multi_filtered else "No se encontró información suficientemente relevante."
    )

    if verbose:
        orig_ids  = {c["meta"]["chunk_id"] for c in (orig_filtered  or [])}
        multi_ids = {c["meta"]["chunk_id"] for c in (multi_filtered or [])}
        overlap   = orig_ids & multi_ids

        print(f"{'═'*70}")
        print(f"QUERY ORIGINAL: {query}")
        print(f"📝 Queries reescritas:")
        for i, q in enumerate(rewritten, 1):
            print(f"  {i}. {q}")
        print(f"📊 Chunks recuperados — original: {len(orig_ids)} | multi-query: {len(multi_ids)} | overlap: {len(overlap)}")

        print(f"{'─'*70}")
        print("RESPUESTA (query original):")
        print(orig_answer)
        print(f"{'─'*70}")
        print("RESPUESTA (multi-query reescrito):")
        print(multi_answer)
        print(f"{'═'*70}")

    return {
        "query": query,
        "rewritten_queries": rewritten,
        "original_chunks":   orig_filtered  or [],
        "multi_chunks":      multi_filtered or [],
        "original_answer":   orig_answer,
        "multi_answer":      multi_answer,
    }


In [20]:
# Prueba rápida — cambia la pregunta y vuelve a ejecutar
DEMO_QUERY = "¿Por qué motivo se vio limitado el avance de producción durante la semana de ejecución de muros de andén?"

result = ask_cosora_rewritten(DEMO_QUERY, n_rewrites=3, verbose=True)


══════════════════════════════════════════════════════════════════════
QUERY ORIGINAL: ¿Por qué motivo se vio limitado el avance de producción durante la semana de ejecución de muros de andén?
📝 Queries reescritas:
  1. muros de andén, avance de producción, limitaciones, ejecución, roles DF, UTE, DO, estado pendiente, incidencia abierta
  2. ¿Qué factores contribuyeron a la restricción del progreso en la construcción de los muros de andén durante la semana de trabajo?
  3. ¿Cuáles fueron las razones técnicas que impidieron el avance en la ejecución de los muros de andén, en términos de encofrado, hormigonado o incidencias en los materiales utilizados?
📊 Chunks recuperados — original: 7 | multi-query: 10 | overlap: 3
──────────────────────────────────────────────────────────────────────
RESPUESTA (query original):
1. Durante la semana de ejecución de los muros de andén, el avance de producción se vio limitado debido a las lluvias, lo que apenas permitió trabajar en las obras (Fuente: 25

In [21]:
# Comparación batch: muestra side-by-side para varias preguntas
COMPARE_QUERIES = [
    # Q1 (easy) — avance_obra
    "¿Qué trabajos se estaban ejecutando en la semana del 29 de enero de 2026?",
    # Q5 (medium) — incidencias
    "¿Qué problema se detectó en el pilar P07 durante la revisión de la estructura metálica?",
    # Q20 (hard) — multi_documento
    "¿Cuáles son los principales factores externos que han impactado el avance de la obra a lo largo del proyecto?",
]

import pandas as pd

rows = []
for q in COMPARE_QUERIES:
    r = ask_cosora_rewritten(q, n_rewrites=3, verbose=False)
    orig_ids  = {c["meta"]["chunk_id"] for c in r["original_chunks"]}
    multi_ids = {c["meta"]["chunk_id"] for c in r["multi_chunks"]}
    rows.append({
        "query": q,
        "chunks_orig":  len(orig_ids),
        "chunks_multi": len(multi_ids),
        "nuevos_chunks": len(multi_ids - orig_ids),
        "rewritten_1":  r["rewritten_queries"][0] if r["rewritten_queries"] else "",
        "rewritten_2":  r["rewritten_queries"][1] if len(r["rewritten_queries"]) > 1 else "",
        "rewritten_3":  r["rewritten_queries"][2] if len(r["rewritten_queries"]) > 2 else "",
    })

df_compare = pd.DataFrame(rows)
print("📊 Comparación de cobertura de chunks (original vs multi-query reescrito):")
display(df_compare[["query", "chunks_orig", "chunks_multi", "nuevos_chunks"]])

print("📝 Queries reescritas generadas:")
display(df_compare[["query", "rewritten_1", "rewritten_2", "rewritten_3"]])

📊 Comparación de cobertura de chunks (original vs multi-query reescrito):


,query,chunks_orig,chunks_multi,nuevos_chunks
0,¿Qué trabajos se estaban ejecutando en la sema...,10,10,6
1,¿Qué problema se detectó en el pilar P07 duran...,10,10,7
2,¿Cuáles son los principales factores externos ...,10,10,5


📝 Queries reescritas generadas:


,query,rewritten_1,rewritten_2,rewritten_3
0,¿Qué trabajos se estaban ejecutando en la sema...,"trabajos, ejecución, semana, 29 enero 2026, ra...",¿Qué actividades se llevaron a cabo en la sema...,"Durante la semana del 29 de enero de 2026, ¿qu..."
1,¿Qué problema se detectó en el pilar P07 duran...,"pilar P07, estructura metálica, revisión, inci...",¿Cuál fue el inconveniente observado en el pil...,¿Qué defecto específico se identificó en el pi...
2,¿Cuáles son los principales factores externos ...,"factores externos, retrasos, clima, permisos, ...",¿Qué elementos externos han influido en el pro...,¿Cuáles son las incidencias y obstáculos exter...


## 5. Evaluación de Retrieval — Sin ground truth

Métricas comparativas entre MrBERT y E5 que **no requieren anotaciones de relevancia ni API externa**.

| Métrica | Qué mide |
|---------|----------|
| **Overlap@K** | Concordancia de chunks entre ambos modelos |
| **RRF Score medio** | Confianza del retrieval (score del top-1) |
| **Diversidad de fuentes** | Nº de documentos distintos en top-K |
| **Cobertura de corpus** | % de chunks únicos alcanzados sobre todas las queries |
| **Similaridad coseno intra-top-K** | Cohesión semántica de los resultados |
| **Latencia** | Velocidad de retrieval |

In [22]:
BENCHMARK_QUERIES = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias AR-29 aparecen?",
    "¿Cuáles son las incidencias más frecuentes en todas las actas?",
    "¿Qué responsable está asignado a las acciones sobre el talud?",
    "¿Qué se acordó sobre hormigonado de zapatas?",
    "Estado de las instalaciones de megafonía",
    "¿Qué documentación debe aportar la UTE sobre estabilidad?",
]

In [23]:
import numpy as np
from collections import Counter
import pandas as pd

EVAL_K = 5  # top-K para las métricas


def eval_retrieval_no_gt(queries, k=EVAL_K):
    """
    Evaluación comparativa de retrieval MrBERT vs E5 SIN ground truth.
    Calcula métricas proxy para cada backend y comparativas entre ambos.
    """
    results = {"mrbert": [], "e5": []}
    all_chunk_ids = {"mrbert": set(), "e5": set()}

    for query in queries:
        for backend in ("mrbert", "e5"):
            t0 = time.perf_counter()
            hits = retrieve_for_backend(query, backend, top_n=k)
            elapsed_ms = (time.perf_counter() - t0) * 1000

            # Chunk IDs y doc IDs
            chunk_ids = [h["meta"]["chunk_id"] for h in hits]
            doc_ids = [h["meta"]["doc_id"] for h in hits]
            scores = [h["score"] for h in hits]

            all_chunk_ids[backend].update(chunk_ids)

            results[backend].append({
                "query": query,
                "chunk_ids": chunk_ids,
                "doc_ids": doc_ids,
                "scores": scores,
                "top1_score": scores[0] if scores else 0,
                "mean_score": np.mean(scores) if scores else 0,
                "n_unique_docs": len(set(doc_ids)),
                "latency_ms": elapsed_ms,
            })

    # --- Métricas por backend ---
    summary_rows = []
    for backend in ("mrbert", "e5"):
        r = results[backend]
        summary_rows.append({
            "backend": backend,
            "avg_top1_rrf_score": np.mean([x["top1_score"] for x in r]),
            "avg_mean_rrf_score": np.mean([x["mean_score"] for x in r]),
            "avg_unique_docs_in_topK": np.mean([x["n_unique_docs"] for x in r]),
            "corpus_coverage (chunks únicos)": len(all_chunk_ids[backend]),
            "avg_latency_ms": np.mean([x["latency_ms"] for x in r]),
        })

    df_summary = pd.DataFrame(summary_rows).set_index("backend")

    # --- Métricas comparativas (Overlap) ---
    overlap_rows = []
    for i, query in enumerate(queries):
        m_ids = set(results["mrbert"][i]["chunk_ids"])
        e_ids = set(results["e5"][i]["chunk_ids"])
        m_docs = set(results["mrbert"][i]["doc_ids"])
        e_docs = set(results["e5"][i]["doc_ids"])

        overlap_chunks = len(m_ids & e_ids) / k if (m_ids or e_ids) else 0
        overlap_docs = len(m_docs & e_docs) / max(len(m_docs | e_docs), 1)

        overlap_rows.append({
            "query": query[:60] + "..." if len(query) > 60 else query,
            f"overlap_chunks@{k}": overlap_chunks,
            f"overlap_docs@{k}": overlap_docs,
            "mrbert_top1_score": results["mrbert"][i]["top1_score"],
            "e5_top1_score": results["e5"][i]["top1_score"],
            "mrbert_unique_docs": results["mrbert"][i]["n_unique_docs"],
            "e5_unique_docs": results["e5"][i]["n_unique_docs"],
        })

    df_overlap = pd.DataFrame(overlap_rows)

    return df_summary, df_overlap, results


# === Ejecutar ===
print("="*70)
print("EVALUACIÓN DE RETRIEVAL — Sin ground truth")
print("="*70)

df_summary, df_overlap, raw_results = eval_retrieval_no_gt(BENCHMARK_QUERIES)

print("\n📊 Resumen por backend:")
display(df_summary)

print(f"\n🔄 Comparación query-por-query (top-{EVAL_K}):")
display(df_overlap)

# Agregados
print("\n--- Agregados ---")
for col in [c for c in df_overlap.columns if c.startswith("overlap")]:
    print(f"  {col} medio: {df_overlap[col].mean():.2%}")
print(f"  Score top-1 medio MrBERT: {df_overlap['mrbert_top1_score'].mean():.4f}")
print(f"  Score top-1 medio E5:     {df_overlap['e5_top1_score'].mean():.4f}")

EVALUACIÓN DE RETRIEVAL — Sin ground truth


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 12491.65it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 14395.33it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 17495.85it/s]
[transformers] ModernBertModel LOAD REPORT fro


📊 Resumen por backend:


,avg_top1_rrf_score,avg_mean_rrf_score,avg_unique_docs_in_topK,corpus_coverage (chunks únicos),avg_latency_ms
backend,,,,,
mrbert,0.026646,0.024097,4.875,27,1455.124300
e5,0.031576,0.029469,4.125,29,4791.937662



🔄 Comparación query-por-query (top-5):


,query,overlap_chunks@5,overlap_docs@5,mrbert_top1_score,e5_top1_score,mrbert_unique_docs,e5_unique_docs
0,¿Qué se decidió sobre el talud?,0.6,0.666667,0.024634,0.033333,5,5
1,¿Cuál es el estado del camino provisional?,0.0,0.333333,0.031778,0.032292,5,3
2,¿Qué incidencias AR-29 aparecen?,0.6,0.428571,0.027778,0.025595,5,5
3,¿Cuáles son las incidencias más frecuentes en ...,0.0,0.000000,0.025712,0.029572,5,3
4,¿Qué responsable está asignado a las acciones ...,0.6,0.666667,0.024645,0.033333,5,5
5,¿Qué se acordó sobre hormigonado de zapatas?,0.0,0.333333,0.027047,0.031818,5,3
6,Estado de las instalaciones de megafonía,0.0,0.000000,0.025645,0.033333,4,4
7,¿Qué documentación debe aportar la UTE sobre e...,0.6,0.428571,0.025926,0.033333,5,5



--- Agregados ---
  overlap_chunks@5 medio: 30.00%
  overlap_docs@5 medio: 35.71%
  Score top-1 medio MrBERT: 0.0266
  Score top-1 medio E5:     0.0316


## 6. Evaluación de Retrieval — LLM-as-Judge (OpenAI)

Métricas clásicas de IR usando GPT-4o-mini como juez de relevancia.

**Cómo funciona:**
1. Para cada query, recuperamos top-K chunks con ambos backends
2. GPT-4o-mini juzga cada chunk: `0` = irrelevante, `1` = parcialmente relevante, `2` = muy relevante
3. Con esos juicios calculamos métricas estándar de Information Retrieval

| Métrica | Qué mide |
|---------|----------|
| **Precision@K** | Fracción de chunks relevantes (score≥1) en top-K |
| **MRR** | Posición del primer resultado relevante (1/rank) |
| **NDCG@K** | Calidad del ranking con relevancia graduada |
| **MAP** | Media de Average Precision por query |

> **Coste estimado:** ~$0.02 (8 queries × 2 backends × 5 chunks × GPT-4o-mini)

> ⚠️ **Requiere:** `GEN_BACKEND="openai"` o `OPENAI_API_KEY` en el `.env`

In [24]:
from openai import OpenAI
from dotenv import load_dotenv
import json

# Cargar API key
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
else:
    load_dotenv(_find_project_root() / ".env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY no encontrada. "
        "Añádela al .env o salta esta sección."
    )

client = OpenAI(api_key=api_key)
LLM_JUDGE_MODEL = "gpt-4o-mini"


def judge_relevance(query: str, chunk_text: str) -> int:
    """
    Pide a GPT-4o-mini que juzgue la relevancia de un chunk para una query.
    Retorna: 0 (irrelevante), 1 (parcialmente relevante), 2 (muy relevante).
    """
    prompt = f"""Eres un evaluador de sistemas de búsqueda de documentos.

Dada la siguiente PREGUNTA y un FRAGMENTO recuperado de un corpus de actas de obra ferroviaria,
evalúa la relevancia del fragmento para responder la pregunta.

Responde SOLO con un número:
- 0 = El fragmento NO contiene información relevante para la pregunta
- 1 = El fragmento contiene información PARCIALMENTE relevante (menciona el tema pero no responde directamente)
- 2 = El fragmento contiene información MUY RELEVANTE (responde directa o sustancialmente a la pregunta)

PREGUNTA: {query}

FRAGMENTO:
{chunk_text[:500]}

PUNTUACIÓN (0, 1 o 2):"""

    response = client.chat.completions.create(
        model=LLM_JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=5,
    )
    answer = response.choices[0].message.content.strip()
    # Extraer el número
    for char in answer:
        if char in "012":
            return int(char)
    return 0  # fallback


print(f"✅ Cliente OpenAI conectado (modelo juez: {LLM_JUDGE_MODEL})")

✅ Cliente OpenAI conectado (modelo juez: gpt-4o-mini)


In [25]:
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain @ k."""
    relevances = relevances[:k]
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))


def ndcg_at_k(relevances, k):
    """Normalized DCG @ k."""
    dcg = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0


def average_precision(relevances):
    """Average Precision (binary: relevance >= 1 cuenta como relevante)."""
    relevant_count = 0
    precision_sum = 0.0
    for i, rel in enumerate(relevances):
        if rel >= 1:
            relevant_count += 1
            precision_sum += relevant_count / (i + 1)
    return precision_sum / relevant_count if relevant_count > 0 else 0.0


def mrr(relevances):
    """Mean Reciprocal Rank (primer resultado con relevance >= 1)."""
    for i, rel in enumerate(relevances):
        if rel >= 1:
            return 1.0 / (i + 1)
    return 0.0


def eval_retrieval_llm_judge(queries, k=EVAL_K):
    """
    Evaluación con LLM-as-Judge para ambos backends.
    Juzga relevancia de cada chunk y calcula Precision@K, MRR, NDCG@K, MAP.
    """
    all_judgments = {"mrbert": [], "e5": []}
    detail_rows = []

    total_calls = len(queries) * 2 * k
    call_count = 0

    for qi, query in enumerate(queries):
        for backend in ("mrbert", "e5"):
            hits = retrieve_for_backend(query, backend, top_n=k)
            relevances = []

            for hi, hit in enumerate(hits):
                call_count += 1
                chunk_text = strip_doc_prefix(hit["text"], backend)
                rel = judge_relevance(query, chunk_text)
                relevances.append(rel)

                detail_rows.append({
                    "query": query[:50] + "..." if len(query) > 50 else query,
                    "backend": backend,
                    "rank": hi + 1,
                    "chunk_id": hit["meta"]["chunk_id"],
                    "doc_id": hit["meta"]["doc_id"],
                    "rrf_score": hit["score"],
                    "llm_relevance": rel,
                })

                if call_count % 10 == 0:
                    print(f"  Progreso: {call_count}/{total_calls} juicios...")

            all_judgments[backend].append({
                "query": query,
                "relevances": relevances,
                f"precision@{k}": sum(1 for r in relevances if r >= 1) / k,
                "mrr": mrr(relevances),
                f"ndcg@{k}": ndcg_at_k(relevances, k),
                "avg_precision": average_precision(relevances),
            })

    # --- Resumen por backend ---
    summary_rows = []
    for backend in ("mrbert", "e5"):
        j = all_judgments[backend]
        summary_rows.append({
            "backend": backend,
            f"Precision@{k}": np.mean([x[f"precision@{k}"] for x in j]),
            "MRR": np.mean([x["mrr"] for x in j]),
            f"NDCG@{k}": np.mean([x[f"ndcg@{k}"] for x in j]),
            "MAP": np.mean([x["avg_precision"] for x in j]),
            "Avg relevance": np.mean([np.mean(x["relevances"]) for x in j]),
        })

    df_summary = pd.DataFrame(summary_rows).set_index("backend")
    df_detail = pd.DataFrame(detail_rows)

    return df_summary, df_detail, all_judgments


# === Ejecutar ===
print("="*70)
print("EVALUACIÓN DE RETRIEVAL — LLM-as-Judge")
print(f"Modelo juez: {LLM_JUDGE_MODEL}")
print(f"Queries: {len(BENCHMARK_QUERIES)}, Backends: 2, Top-K: {EVAL_K}")
print(f"Total llamadas API: {len(BENCHMARK_QUERIES) * 2 * EVAL_K}")
print("="*70)

df_llm_summary, df_llm_detail, llm_judgments = eval_retrieval_llm_judge(BENCHMARK_QUERIES)

print("\n🏆 Resumen de métricas IR por backend:")
display(df_llm_summary)

print(f"\n📝 Detalle de juicios (primeras 20 filas):")
display(df_llm_detail.head(20))

EVALUACIÓN DE RETRIEVAL — LLM-as-Judge
Modelo juez: gpt-4o-mini
Queries: 8, Backends: 2, Top-K: 5
Total llamadas API: 80


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 33478.48it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13667.82it/s]


  Progreso: 10/80 juicios...


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 15686.21it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15451.92it/s]


  Progreso: 20/80 juicios...


...

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 12606.53it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15307.12it/s]


  Progreso: 80/80 juicios...

🏆 Resumen de métricas IR por backend:


,Precision@5,MRR,NDCG@5,MAP,Avg relevance
backend,,,,,
mrbert,0.55,0.7500,0.763358,0.723437,0.800
e5,0.70,0.8375,0.865879,0.859896,1.075



📝 Detalle de juicios (primeras 20 filas):


,query,backend,rank,chunk_id,doc_id,rrf_score,llm_relevance
0,¿Qué se decidió sobre el talud?,mrbert,1,243591-DO-AVO-11-V07-251210__c0000,243591-DO-AVO-11-V07-251210,0.024634,0
1,¿Qué se decidió sobre el talud?,mrbert,2,254275-DO-AVO-14-V07__c0023,254275-DO-AVO-14-V07,0.024359,2
2,¿Qué se decidió sobre el talud?,mrbert,3,254275-DO-AVO-15-V07__c0023,254275-DO-AVO-15-V07,0.024027,2
3,¿Qué se decidió sobre el talud?,mrbert,4,254275-DO-AVO-16-V07__c0020,254275-DO-AVO-16-V07,0.023705,2
4,¿Qué se decidió sobre el talud?,mrbert,5,244170-DOB-AVO-07-V00-A0__c0007,244170-DOB-AVO-07-V00-A0,0.023559,0
5,¿Qué se decidió sobre el talud?,e5,1,254275-DO-AVO-14-V07__c0023,254275-DO-AVO-14-V07,0.033333,2
6,¿Qué se decidió sobre el talud?,e5,2,254275-DO-AVO-15-V07__c0023,254275-DO-AVO-15-V07,0.032787,2
7,¿Qué se decidió sobre el talud?,e5,3,254275-DO-AVO-16-V07__c0020,254275-DO-AVO-16-V07,0.032258,2
8,¿Qué se decidió sobre el talud?,e5,4,254275-DO-AVO-22-V01__c0036,254275-DO-AVO-22-V01,0.029572,1
9,¿Qué se decidió sobre el talud?,e5,5,243591-DO-AVO-11-V07-251210__c0001,243591-DO-AVO-11-V07-251210,0.023449,0


## 6b. LLM-as-Judge: Original Query vs Query Rewriting

Mismo backend (`EMBED_BACKEND`), misma métrica, **dos estrategias de retrieval**:

| Estrategia | Retrieval |
|---|---|
| **Original** | `retrieve_active(query)` — una sola query |
| **Reescrito** | `retrieve_multi_query([query] + rewrite_query(query, n=3))` — 4 queries fusionadas con RRF |

El LLM judge puntúa cada chunk (0 / 1 / 2) y calculamos Precision@K, MRR, NDCG@K y MAP.  
La columna **Δ** muestra la mejora neta del rewriting: **verde** = mejora, **rojo** = empeora.

In [28]:
def eval_rewriting_llm_judge(queries, k=EVAL_K):
    """
    Compara retrieval SIN vs CON query rewriting usando LLM-as-Judge.
    Fija el backend activo (EMBED_BACKEND).
    Devuelve un DataFrame con métricas por query y columnas Δ.
    """
    rows = []
    total_calls = len(queries) * 2 * k
    call_count = 0

    for query in queries:
        # ── Retrieval original (una sola query) ──────────────────────────
        orig_hits = retrieve_active(query, top_n=k)
        orig_rels = []
        for hit in orig_hits:
            call_count += 1
            text = strip_doc_prefix(hit["text"], EMBED_BACKEND)
            orig_rels.append(judge_relevance(query, text))
            if call_count % 10 == 0:
                print(f"  [{call_count}/{total_calls}] juicios completados…")

        # ── Retrieval multi-query (original + 3 reescritas) ──────────────
        rewritten = rewrite_query(query, n=3)
        multi_hits = retrieve_multi_query([query] + rewritten, top_n=k)
        multi_rels = []
        for hit in multi_hits:
            call_count += 1
            text = strip_doc_prefix(hit["text"], EMBED_BACKEND)
            multi_rels.append(judge_relevance(query, text))
            if call_count % 10 == 0:
                print(f"  [{call_count}/{total_calls}] juicios completados…")

        rows.append({
            "query":              query[:52] + "…" if len(query) > 52 else query,
            f"P@{k} orig":        round(sum(r >= 1 for r in orig_rels)  / k, 3),
            f"P@{k} rew":         round(sum(r >= 1 for r in multi_rels) / k, 3),
            f"ΔP@{k}":            round(sum(r >= 1 for r in multi_rels) / k
                                        - sum(r >= 1 for r in orig_rels)  / k, 3),
            f"NDCG@{k} orig":     round(ndcg_at_k(orig_rels,  k), 3),
            f"NDCG@{k} rew":      round(ndcg_at_k(multi_rels, k), 3),
            f"ΔNDCG@{k}":         round(ndcg_at_k(multi_rels, k) - ndcg_at_k(orig_rels, k), 3),
            "MRR orig":           round(mrr(orig_rels),  3),
            "MRR rew":            round(mrr(multi_rels), 3),
            "ΔMRR":               round(mrr(multi_rels) - mrr(orig_rels), 3),
            "MAP orig":           round(average_precision(orig_rels),  3),
            "MAP rew":            round(average_precision(multi_rels), 3),
            "ΔMAP":               round(average_precision(multi_rels) - average_precision(orig_rels), 3),
        })

    return pd.DataFrame(rows)


# ── Ejecución ─────────────────────────────────────────────────────────
print("=" * 70)
print("LLM-as-Judge: ORIGINAL vs QUERY REWRITING")
print(f"Backend: {EMBED_BACKEND}  |  Juez: {LLM_JUDGE_MODEL}  |  Top-K: {EVAL_K}")
print(f"Queries: {len(BENCHMARK_QUERIES)}  |  Llamadas API estimadas: {len(BENCHMARK_QUERIES) * 2 * EVAL_K}")
print("=" * 70)

df_rew = eval_rewriting_llm_judge(BENCHMARK_QUERIES)

# ── Tabla por query ────────────────────────────────────────────────────
def _color_delta(v):
    if isinstance(v, float):
        if v > 0:   return "color: green; font-weight: bold"
        if v < 0:   return "color: red"
    return ""

delta_cols = [c for c in df_rew.columns if c.startswith("Δ")]

print(f"\n📊 Métricas por query — original vs reescrito (Top-{EVAL_K}):")
display(
    df_rew.style
    .format({c: "{:+.3f}" for c in delta_cols} |
            {c: "{:.3f}" for c in df_rew.columns if c not in delta_cols and c != "query"})
    .map(_color_delta, subset=delta_cols)
    .set_properties(**{"text-align": "center"})
    .set_properties(subset=["query"], **{"text-align": "left"})
)

# ── Resumen global ─────────────────────────────────────────────────────
print("\n🏆 Resumen global (media sobre todas las queries):")

summary_data = {
    "Estrategia":    ["Original", "Reescrito", "Δ (mejora)"],
    f"Precision@{EVAL_K}": [
        df_rew[f"P@{EVAL_K} orig"].mean(),
        df_rew[f"P@{EVAL_K} rew"].mean(),
        df_rew[f"ΔP@{EVAL_K}"].mean(),
    ],
    f"NDCG@{EVAL_K}": [
        df_rew[f"NDCG@{EVAL_K} orig"].mean(),
        df_rew[f"NDCG@{EVAL_K} rew"].mean(),
        df_rew[f"ΔNDCG@{EVAL_K}"].mean(),
    ],
    "MRR": [
        df_rew["MRR orig"].mean(),
        df_rew["MRR rew"].mean(),
        df_rew["ΔMRR"].mean(),
    ],
    "MAP": [
        df_rew["MAP orig"].mean(),
        df_rew["MAP rew"].mean(),
        df_rew["ΔMAP"].mean(),
    ],
}
df_global = pd.DataFrame(summary_data).set_index("Estrategia")
display(
    df_global.style
    .format("{:+.4f}", subset=pd.IndexSlice[["Δ (mejora)"], :])
    .format("{:.4f}",  subset=pd.IndexSlice[["Original", "Reescrito"], :])
    .map(lambda v: "color: green; font-weight: bold" if isinstance(v, float) and v > 0
                   else ("color: red" if isinstance(v, float) and v < 0 else ""),
         subset=pd.IndexSlice[["Δ (mejora)"], :])
    .set_caption(f"Backend: {EMBED_BACKEND}  ·  Juez: {LLM_JUDGE_MODEL}  ·  Top-{EVAL_K}")
)

LLM-as-Judge: ORIGINAL vs QUERY REWRITING
Backend: e5  |  Juez: gpt-4o-mini  |  Top-K: 5
Queries: 8  |  Llamadas API estimadas: 80


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14191.15it/s]


  [10/80] juicios completados…
  [20/80] juicios completados…
  [30/80] juicios completados…
  [40/80] juicios completados…
  [50/80] juicios completados…
  [60/80] juicios completados…
  [70/80] juicios completados…
  [80/80] juicios completados…

📊 Métricas por query — original vs reescrito (Top-5):


,query,P@5 orig,P@5 rew,ΔP@5,NDCG@5 orig,NDCG@5 rew,ΔNDCG@5,MRR orig,MRR rew,ΔMRR,MAP orig,MAP rew,ΔMAP
0,¿Qué se decidió sobre el talud?,0.800,0.800,+0.000,1.000,0.991,-0.009,1.000,1.000,+0.000,1.000,0.950,-0.050
1,¿Cuál es el estado del camino provisional?,1.000,1.000,+0.000,0.891,0.856,-0.035,1.000,1.000,+0.000,1.000,1.000,+0.000
2,¿Qué incidencias AR-29 aparecen?,0.600,0.600,+0.000,1.000,1.000,+0.000,1.000,1.000,+0.000,1.000,1.000,+0.000
3,¿Cuáles son las incidencias más frecuentes en todas …,0.200,0.400,+0.200,0.387,0.571,+0.184,0.200,0.333,+0.133,0.200,0.417,+0.217
4,¿Qué responsable está asignado a las acciones sobre …,0.600,0.600,+0.000,1.000,1.000,+0.000,1.000,1.000,+0.000,1.000,1.000,+0.000
5,¿Qué se acordó sobre hormigonado de zapatas?,0.800,1.000,+0.200,0.761,1.000,+0.239,0.500,1.000,+0.500,0.679,1.000,+0.321
6,Estado de las instalaciones de megafonía,1.000,1.000,+0.000,1.000,0.956,-0.044,1.000,1.000,+0.000,1.000,1.000,+0.000
7,¿Qué documentación debe aportar la UTE sobre estabil…,0.600,0.800,+0.200,1.000,1.000,+0.000,1.000,1.000,+0.000,1.000,1.000,+0.000



🏆 Resumen global (media sobre todas las queries):


,Precision@5,NDCG@5,MRR,MAP
Estrategia,,,,
Original,0.7000,0.8799,0.8375,0.8599
Reescrito,0.7750,0.9218,0.9166,0.9209
Δ (mejora),+0.0750,+0.0419,+0.0791,+0.0610


## 7. Pruebas Interactivas del Pipeline Final

Ahora que hemos evaluado ambos modelos en las secciones anteriores, puedes elegir tu ganador y probar el pipeline completo (Búsqueda + Generador LLM).

Puedes cambiar la variable `MODELO_GANADOR` directamente aquí abajo, sin tener que volver al principio del notebook.

In [16]:
# Escribe "mrbert" o "e5" según lo que hayas decidido en los benchmarks
MODELO_GANADOR = "e5"

MIS_PREGUNTAS = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    # "Añade más preguntas aquí...",
]

print(f"Probando pipeline completo con modelo: {MODELO_GANADOR}\n")
print("=" * 80)

for pregunta in MIS_PREGUNTAS:
    print(f"\n\n👤 PREGUNTA: {pregunta}")
    print("-" * 80)
    
    respuesta = ask_cosora(pregunta, verbose=False, backend=MODELO_GANADOR)
    
    print(f"🤖 RESPUESTA COSORA:\n{respuesta}")
    print("=" * 80)

Probando pipeline completo con modelo: e5



👤 PREGUNTA: ¿Qué se decidió sobre el talud?
--------------------------------------------------------------------------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10475.76it/s]


🤖 RESPUESTA COSORA:
1. La Dirección Facultativa (DF) solicitó a la Unión Temporal de Empresas (UTE) información adicional para cerrar la verificación de la ejecución de hinca de perfiles en relación con la estabilidad del talud. Específicamente, se requiere confirmar las profundidades reales de hinca, las separaciones y la orientación respecto al talud calculado (Fuente: 254275-DO-AVO-14-V07, 254275-DO-AVO-15-V07, 254275-DO-AVO-16-V07).


👤 PREGUNTA: ¿Cuál es el estado del camino provisional?
--------------------------------------------------------------------------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10371.62it/s]


🤖 RESPUESTA COSORA:
1. **Condiciones del Camino Provisional**: El camino provisional ha sido afectado por lluvias abundantes, lo que ha generado una saturación significativa del terreno y aumentado el riesgo de desplazamientos laterales debido al empuje de las tierras húmedas (Fuente: 254275-DO-AVO-15-V07).

2. **Acciones de Mejora**: Se ha comenzado a rebajar el nivel de tierras del camino provisional. Esta acción tiene como objetivo mejorar las condiciones de seguridad y estabilidad del entorno al disminuir el empuje de tierras hacia las parcelas (Fuente: 254275-DO-AVO-14-V07, 254275-DO-AVO-19-V07).

3. **Recomendaciones Futuras**: Se solicita que, en la medida de lo posible, se continúe con el rebaje del nivel de tierras del camino provisional como medida preventiva (Fuente: 254275-DO-AVO-23-V01).

4. **Documentación Pendiente**: La Dirección Facultativa ha solicitado a la UTE que aporte la documentación gráfica que defina el camino provisional ejecutado (Fuente: 254275-DO-AVO-14-V0

In [41]:
# Ejecuta esta celda para tener un chat interactivo por consola.
# Recuerda que usa el MODELO_GANADOR que definiste arriba.
# Escribe "salir" para terminar.

while True:
    q = input("\nEscribe tu pregunta para COSORA (o \'salir\'): ")
    if q.lower() in ['salir', 'exit', 'quit']:
        print("¡Hasta luego!")
        break
    if not q.strip():
        continue
        
    print(f"\nBuscando en actas con {MODELO_GANADOR}...")
    resp = ask_cosora(q, verbose=False, backend=MODELO_GANADOR)
    print("\n🤖 RESPUESTA:")
    print(resp)
    print("-" * 60)


Buscando en actas con e5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🤖 RESPUESTA:
1. La Dirección Facultativa (DF) solicitó a la UTE información adicional para cerrar la verificación de la ejecución de hinca de perfiles en el talud. Específicamente, se requiere confirmar las profundidades reales de hinca, las separaciones y la orientación respecto al talud calculado (Fuente: 254275-DO-AVO-15-V07, 254275-DO-AVO-14-V07, 254275-DO-AVO-16-V07).

2. Además, se solicitó a la UTE que aporte los Certificados de material de hinca, incluyendo el tipo de carril, acero y sus características. El Anejo de cálculo modela hincas de carril 54E1 (Fuente: 254275-DO-AVO-14-V07, 254275-DO-AVO-16-V07).

3. Durante una visita de obra, se detectó un tramo de solera en voladizo en la parte superior del talud de excavación que no contaba con el apuntalamiento necesario. Se indicó a la constructora que debía proceder de manera inmediata a apuntalar correctamente esta solera para garantizar su estabilidad temporal (Fuente: 254275-DO-AVO-22-V01).
----------------------------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🤖 RESPUESTA:
1. La Dirección Facultativa (DF) ha solicitado a la UTE que aporte la documentación gráfica que defina el camino provisional ejecutado (Fuente: 254275-DO-AVO-14-V07, 254275-DO-AVO-21-V01, 254275-DO-AVO-17-V07, 254275-DO-AVO-15-V07, 254275-DO-AVO-19-V07, 254275-DO-AVO-16-V07, 254275-DO-AVO-20-V07).

2. Se está llevando a cabo el rebaje del nivel de tierras del camino provisional, lo cual mejora las condiciones de seguridad y estabilidad del entorno al disminuir el empuje de tierras hacia las parcelas (Fuente: 254275-DO-AVO-19-V07).

3. Se ha generado una saturación significativa del terreno en el ámbito del camino provisional, aumentando el riesgo de desplazamientos laterales. Por ello, se ha solicitado a la constructora que intensifique la inspección y control de las barreras tipo New Jersey (Fuente: 254275-DO-AVO-15-V07).

4. La constructora ha comenzado a bajar el nivel de tierras del camino provisional para facilitar la reducción de la presión de las tierras en el nive

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🤖 RESPUESTA:
1. **Trabajos en la marquesina de la vía 01:**
   - Se han realizado trabajos de eliminación de la chapa de cubierta existente y de las correas, excepto la correa límite con la vía (Fuente: 244170-DOB-AVO-09-V00, 244170-DOB-AVO-10-V00-A0).
   - Se ha instalado la nueva correa y se ha comprobado la posibilidad de instalar el canalón de recogida de aguas y el panel sándwich (Fuente: 244170-DOB-AVO-12-V00-A0).
   - Se ha iniciado la instalación del Descargador de intervalos (Fuente: 244170-DOB-AVO-12-V00-A0).

2. **Trabajos en la marquesina de la vía 02:**
   - Se han realizado trabajos de aplicación de desoxidado y limpieza de la estructura metálica (Fuente: 244170-DOB-AVO-05-V00-A0).

3. **Cubiertas planas:**
   - Se ha finalizado la instalación de la capa de manta de geotextil y protección de grava (Fuente: 244170-DOB-AVO-09-V00).

4. **Reuniones y visitas de obra:**
   - Se han realizado varias visitas de obra con la asistencia de la Dirección Facultativa y la Constructo